In [ ]:
import sys
import math
import importlib

sys.path.append("..")

import keypoint_detectors
importlib.reload(keypoint_detectors)
from keypoint_detectors import FastDetector, SuperPointDetector, ShiTomasiDetector, HarrisDetector

import conf
importlib.reload(conf)

import core.QuadtreeWSIPairDataset
importlib.reload(core.QuadtreeWSIPairDataset)
from core.QuadtreeWSIPairDataset import QuadtreeWSIPairDataset

import torch
import numpy as np
import matplotlib.pyplot as plt

ANNOTATION_PATH = conf.ANNOTATION_PATH
CNN_INPUT_HEIGHT = conf.CNN_INPUT_HEIGHT
CNN_INPUT_WIDTH = conf.CNN_INPUT_WIDTH

dataset = QuadtreeWSIPairDataset(
    annotation_path=ANNOTATION_PATH,
    input_height=CNN_INPUT_HEIGHT,
    input_width=CNN_INPUT_WIDTH,
)

unique_pair_ids = sorted({job["pair_id"] for job in dataset.tile_jobs})
print(f"Dataset: {len(dataset)} tiles across {len(unique_pair_ids)} image pairs")


def tensor_to_numpy_gray(t):
    return t.squeeze(0).detach().cpu().numpy()

def tensor_to_numpy_rgb(t):
    return t.detach().cpu().permute(1, 2, 0).numpy()


In [ ]:
def compare_detectors(
    pair_id_index,
    image_side,
    detectors,
    quadtree_level,
    x_idx,
    y_idx,
    color=True,
):
    if pair_id_index < 0 or pair_id_index >= len(unique_pair_ids):
        raise ValueError(
            f"pair_id_index {pair_id_index} out of range [0, {len(unique_pair_ids) - 1}]"
        )
    pair_id = unique_pair_ids[pair_id_index]

    sample_idx = next(
        (
            i for i, job in enumerate(dataset.tile_jobs)
            if job["pair_id"] == pair_id
            and job["crop_depth"] == quadtree_level
            and job["x_idx"] == x_idx
            and job["y_idx"] == y_idx
        ),
        None,
    )

    if sample_idx is None:
        raise ValueError(
            f"No tile found: pair_id={pair_id}, level={quadtree_level}, x={x_idx}, y={y_idx}"
        )

    sample = dataset[sample_idx]
    meta = sample["meta"]

    gray = tensor_to_numpy_gray(sample[image_side])
    rgb = tensor_to_numpy_rgb(sample[f"{image_side}_vis"])

    display_image = rgb if color else gray
    cmap = None if color else "gray"

    n = len(detectors)
    n_cols = 2
    n_rows = math.ceil(n / n_cols)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * 6, n_rows * 5),
        squeeze=False,
    )

    print(
        f"Pair index {pair_id_index}  |  pair_id={pair_id}  |  "
        f"source={meta['source_image_id']} → target={meta['target_image_id']}  |  "
        f"{image_side}  |  level={quadtree_level}, grid={meta['grid']}, tile=({x_idx},{y_idx})"
    )

    for i, detector in enumerate(detectors):
        ax = axes[i // n_cols][i % n_cols]
        pts = detector.detect(gray)
        n_kp = pts.shape[1] if pts.ndim == 2 else 0

        ax.imshow(display_image, cmap=cmap)
        ax.set_title(f"{detector.name}  |  {n_kp} keypoints", fontsize=11)
        ax.axis("off")

        if n_kp > 0:
            ax.scatter(
                pts[0], pts[1],
                s=8, facecolors="none", edgecolors="r", linewidths=0.6,
            )

    for i in range(n, n_rows * n_cols):
        axes[i // n_cols][i % n_cols].set_visible(False)

    plt.tight_layout()
    plt.show()


In [ ]:
compare_detectors(
    pair_id_index=0,
    image_side="fixed",
    detectors=[
        SuperPointDetector(),
        FastDetector(),
        ShiTomasiDetector(),
        HarrisDetector(),
    ],
    quadtree_level=4,
    x_idx=0,
    y_idx=0,
    color=True,
)
